# Regulatory Document Q&A Agent

RAG-based agent that answers compliance questions from a PDF and cites the exact page it pulled the answer from.

Steps: load PDF -> split into chunks -> embed -> store in vector DB -> retrieve -> generate answer with citation.

In [1]:
!pip install -q langchain langchain-community langchain-google-genai
!pip install -q langchain-text-splitters langchain-huggingface langchain-chroma
!pip install -q pypdf chromadb sentence-transformers
!pip install -q gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import getpass

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("libraries loaded")

/tmp/ipykernel_450/1308518677.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


libraries loaded


In [25]:
os.environ["GOOGLE_API_KEY"] = getpass.getpass("enter your gemini api key: ")

enter your gemini api key: ··········


In [26]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("file received:", pdf_path)

Saving fintech (1).pdf to fintech (1) (1).pdf
file received: fintech (1) (1).pdf


In [27]:
loader = PyPDFLoader(pdf_path)
doc_pages = loader.load()

print("number of pages:", len(doc_pages))
print(doc_pages[0].metadata)

number of pages: 8
{'producer': 'Adobe PDF Library 21.5.92', 'creator': 'Acrobat PDFMaker 21 for Word', 'creationdate': '2026-07-17T11:00:28+02:00', 'author': 'Rajat Chauhan', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-07-17T11:15:33+02:00', 'sourcemodified': 'D:20260615140409', 'subject': '', 'title': '', 'source': 'fintech (1) (1).pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}


## document loading check

In [28]:
print(doc_pages[0].page_content[:500])

In [29]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

doc_chunks = splitter.split_documents(doc_pages)
print("total chunks:", len(doc_chunks))

total chunks: 25


In [30]:
c = doc_chunks[5]
print("page:", c.metadata.get("page"))
print(c.page_content[:300])

page: 2
Only secondary data have been used in this report. The primary sources are Taptap Send's 
official website and help center where they provided all these information for the company: 
the scope of services, reached destinations, their launch history, and their financial model. 
They are used to gain 


## embeddings and vector store

In [31]:
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("embedding model ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding model ready


In [32]:
vector_db = Chroma.from_documents(
    documents=doc_chunks,
    embedding=embedder
)

retriever = vector_db.as_retriever(search_kwargs={"k": 4})
print("vector store built (in-memory)")

vector store built (in-memory)


In [33]:
check_query = "what is this document mainly about"
found = retriever.invoke(check_query)

for i, d in enumerate(found):
    print(f"result {i+1} - page {d.metadata.get('page')}")
    print(d.page_content[:180])
    print()

result 1 - page 6
and it demands operational resiliency to be managed. This type of project is most suitable for 
group projects, in which one group member is in charge of researching, another of wr

result 2 - page 6
and it demands operational resiliency to be managed. This type of project is most suitable for 
group projects, in which one group member is in charge of researching, another of wr

result 3 - page 2
Only secondary data have been used in this report. The primary sources are Taptap Send's 
official website and help center where they provided all these information for the company

result 4 - page 2
Only secondary data have been used in this report. The primary sources are Taptap Send's 
official website and help center where they provided all these information for the company



## rag chain with citation

In [16]:
prompt_text = """You are a compliance assistant. Use only the context given below to answer.

Rules:
- Do not use any outside knowledge, only the context.
- If the context does not contain the answer, reply: "This is not covered in the provided document."
- Identify the section name or number the answer comes from, if the context shows one (e.g., a heading, clause number, or article number). If no section marker is visible in the context, write "Section: not labeled in document".
- End every answer in this exact format:
  [Source: Section - <section>, Page <page number>]
- Keep the answer short and factual.

Context:
{context}

Question: {question}

Answer:"""

qa_prompt = PromptTemplate(template=prompt_text, input_variables=["context", "question"])

In [17]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

def format_docs(docs):
    return "\n\n".join(f"[Page {d.metadata.get('page')}] {d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | qa_prompt
    | llm
    | StrOutputParser()
)

print("chain ready")

chain ready


In [18]:
def ask(question):
    answer = rag_chain.invoke(question)
    source_docs = retriever.invoke(question)
    pages_used = sorted(set(d.metadata.get("page") for d in source_docs))

    print("Q:", question)
    print("A:", answer)
    print("pages used:", pages_used)
    print("-" * 50)
    return answer

## testing

In [19]:
ask("What is the main purpose of this document?")

Q: What is the main purpose of this document?
A: The report examines Taptap Send's business model, market position, and growth environment to analyze the innovation of financial inclusion and international payments.

[Source: Section - 2 Introduction & rationale, Page 2]
pages used: [2, 6]
--------------------------------------------------


"The report examines Taptap Send's business model, market position, and growth environment to analyze the innovation of financial inclusion and international payments.\n\n[Source: Section - 2 Introduction & rationale, Page 2]"

In [20]:
ask("What are the key requirements mentioned in this document?")

Q: What are the key requirements mentioned in this document?
A: The key regulatory requirements mentioned are KYC, AML, screening for sanctions, licensing, and consumer protection.

[Source: Section - 11.5 Legal, Page 6]
pages used: [1, 4, 6]
--------------------------------------------------


'The key regulatory requirements mentioned are KYC, AML, screening for sanctions, licensing, and consumer protection.\n\n[Source: Section - 11.5 Legal, Page 6]'

In [21]:
ask("What is the population of Mars?")

Q: What is the population of Mars?
A: This is not covered in the provided document.

[Source: Section - not labeled in document, Page not applicable]
pages used: [1, 3, 7]
--------------------------------------------------


'This is not covered in the provided document.\n\n[Source: Section - not labeled in document, Page not applicable]'

In [22]:
import gradio as gr

def gradio_ask(question):
    answer = rag_chain.invoke(question)
    source_docs = retriever.invoke(question)
    pages_used = sorted(set(d.metadata.get("page") for d in source_docs))
    return answer + "\n\npages used: " + str(pages_used)

demo = gr.Interface(
    fn=gradio_ask,
    inputs=gr.Textbox(label="ask something about the document"),
    outputs=gr.Textbox(label="answer"),
    title="Regulatory Document Q&A Agent"
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dbf36f59d4989e68ee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://dbf36f59d4989e68ee.gradio.live


## conclusion

page metadata survives chunking correctly, so citations stay accurate. the prompt reliably forces the model to either answer with a page number or state clearly that the info isn't in the document. main limitation is scanned PDFs - they'd need OCR added since this only reads selectable text.
